# SIC 2026 · Inteligencia Artificial
## Clase 17: Autoencoders, GAN y primeros pasos con Hugging Face (Capítulos 9 y 10)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Cerramos el Capítulo 9 con las redes que **reconstruyen y generan**, y abrimos el Capítulo 10: los **modelos preentrenados** de Hugging Face, que son la respuesta a lo que venimos viendo desde la Clase 13: las redes necesitan muchísimos datos, y estos modelos ya los vieron.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:40 | 1. Autoencoder | Cap. 9, 82 a 89 | Comprimir prendas a 32 números y reconstruirlas |
| 18:40 a 19:10 | 2. Usos del autoencoder | | Quitar ruido y **detectar anomalías** |
| 19:10 a 19:40 | 3 y 4. GAN | Cap. 9, 91 a 132 | La idea, y verla generar prendas |
| 19:40 a 19:50 | Pausa | | |
| 19:50 a 20:15 | 5. Hugging Face | Cap. 10, 12 a 20 | El Hub: modelos, datasets y Spaces |
| 20:15 a 21:00 | 6. Pipeline API | Cap. 10, 21 a 39 | Qué pasa por dentro y un recorrido por tareas en español |
| 21:00 a 21:55 | 7. Preentrenado contra lo nuestro | | IMDb y las reseñas de Samsung, comparados con nuestros modelos |
| 21:55 a 22:00 | Cierre | | |

**Antes de empezar:** *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*.

Hay 12 ejercicios con verificación; las soluciones están al final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import time, warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

def mostrar(filas, titulos=None, n=8, alto=1.6):
    """Muestra varias filas de imágenes de 28 x 28, una fila por lista."""
    fig, ejes = plt.subplots(len(filas), n, figsize=(n * 1.3, alto * len(filas)))
    ejes = np.atleast_2d(ejes)
    for f, imagenes in enumerate(filas):
        for k in range(n):
            ejes[f, k].imshow(np.asarray(imagenes[k]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            ejes[f, k].axis("off")
        if titulos: ejes[f, 0].set_title(titulos[f], fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = (X_tr / 255.0).astype("float32").reshape(-1, 784)
X_te = (X_te / 255.0).astype("float32").reshape(-1, 784)
PRENDAS = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo", "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]
print("Entrenamiento:", X_tr.shape, "| Prueba:", X_te.shape, "| GPU:", tf.config.list_physical_devices("GPU") or "no")

---
# Parte 1. Autoencoder (láminas 82 a 89)

> **Autoencoder (codificador automático):** una red que aprende a **copiar su entrada en su salida**, pero obligada a pasar por un **cuello de botella** de pocos números.

Parece una tarea tonta: ¿para qué copiar una imagen? El truco está en el cuello de botella. Para reconstruir 784 pixeles a partir de solo 32 números, la red tiene que aprender **lo esencial** de las imágenes: qué forma tiene una camiseta, qué distingue a un zapato.

| Parte | Qué hace | En nuestra red |
|---|---|---|
| **Codificador** (encoder) | Comprime la imagen en pocos números | 784 → 128 → 32 |
| **Código** (espacio latente) | La versión comprimida: 32 números que resumen la prenda | 32 |
| **Decodificador** (decoder) | Reconstruye la imagen desde el código | 32 → 128 → 784 |

**No necesita etiquetas:** la respuesta correcta es la misma imagen. Es aprendizaje **no supervisado**, como el clustering del Capítulo 6.

## Paso 1.1. Construir el autoencoder

Lo armamos como dos modelos (codificador y decodificador) unidos, para poder usar cada mitad por separado después.

In [ ]:
CODIGO = 32

keras.utils.set_random_seed(42)
codificador = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(CODIGO, activation="relu"),           # el cuello de botella
], name="codificador")

decodificador = keras.Sequential([
    layers.Input(shape=(CODIGO,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(784, activation="sigmoid"),           # sigmoide: pixeles entre 0 y 1
], name="decodificador")

autoencoder = keras.Sequential([codificador, decodificador], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

**Qué deberías ver:** dos bloques, el codificador y el decodificador, con cerca de 210.000 parámetros en total (casi todos en las dos capas que conectan con los 784 pixeles).

**¿Por qué sigmoide y `binary_crossentropy`?** Cada pixel está entre 0 y 1, como una probabilidad. La sigmoide asegura salidas en ese rango, y la entropía cruzada compara pixel por pixel. También funciona con `"mse"`.

### Ejercicio 1
¿Cuántas veces más pequeño es el código que la imagen? Guarda en `factor_compresion` la división entre los pixeles de la imagen y los números del código.

In [ ]:
factor_compresion = None  # TU CÓDIGO AQUÍ
print(factor_compresion)

In [ ]:
revisar(lambda: np.isclose(factor_compresion, 784 / 32))

## Paso 1.2. Entrenar

Fíjense en el `fit`: **la entrada y la respuesta son lo mismo**, `X_tr` y `X_tr`.

In [ ]:
historia = autoencoder.fit(X_tr, X_tr, epochs=10, batch_size=256, validation_data=(X_te, X_te), verbose=0)
print("Pérdida final en validación:", round(historia.history["val_loss"][-1], 4))

reconstruidas = autoencoder.predict(X_te[:8], verbose=0)
mostrar([X_te[:8], reconstruidas], ["Originales", "Reconstruidas desde 32 números"])

**Qué deberías ver:** reconstrucciones reconocibles pero **borrosas**: se mantiene la forma de cada prenda y se pierden detalles finos (estampados, texturas). Es el precio de comprimir 24 veces.

### Ejercicio 2
Usa solo el **codificador** para obtener los códigos de las 8 primeras imágenes de prueba, y guárdalos en `codigos_8`. ¿Qué forma tienen?

In [ ]:
codigos_8 = None  # TU CÓDIGO AQUÍ
print(codigos_8.shape if codigos_8 is not None else None)

In [ ]:
revisar(lambda: codigos_8.shape == (8, CODIGO))

## Paso 1.3. ¿El código guarda información útil?

Si el código resume lo esencial, prendas del mismo tipo deberían tener códigos parecidos. Lo comprobamos de dos formas: **dibujando** los códigos en 2D (con PCA, Clase 8) y **clasificando** con ellos.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

codigos_tr = codificador.predict(X_tr, verbose=0)
codigos_te = codificador.predict(X_te, verbose=0)

Z = PCA(n_components=2).fit_transform(codigos_te[:3000])
plt.figure(figsize=(8, 5.5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=y_te[:3000], cmap="tab10", s=6)
plt.legend(sc.legend_elements()[0], PRENDAS, fontsize=7, loc="best"); plt.title("Códigos de 32 números, vistos en 2D")
plt.show()

exac_codigo = accuracy_score(y_te, LogisticRegression(max_iter=2000).fit(codigos_tr, y_tr).predict(codigos_te))
print(f"Logística con los 32 números del código: {exac_codigo:.3f}")

**Qué deberías ver:** grupos de colores separados: zapatos por un lado, pantalones por otro, y las prendas superiores mezcladas entre sí. Una logística con solo los 32 números del código llega a cerca de **0,8**, aunque el autoencoder **nunca vio las etiquetas**.

Esa es la utilidad de fondo: **el codificador aprendió una representación compacta y con sentido**, como los embeddings de palabras de la Clase 13, pero para imágenes.

---
# Parte 2. Usos del autoencoder

## Paso 2.1. Quitar ruido (denoising autoencoder)

**La idea:** entrenar con imágenes **con ruido** como entrada, y las **limpias** como respuesta. La red aprende a separar la prenda del ruido.

In [ ]:
rng = np.random.default_rng(0)
ruido = 0.4
X_tr_ruido = np.clip(X_tr + ruido * rng.normal(size=X_tr.shape), 0, 1).astype("float32")
X_te_ruido = np.clip(X_te + ruido * rng.normal(size=X_te.shape), 0, 1).astype("float32")

keras.utils.set_random_seed(42)
limpiador = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation="relu"), layers.Dense(64, activation="relu"),
    layers.Dense(256, activation="relu"), layers.Dense(784, activation="sigmoid"),
])
limpiador.compile(optimizer="adam", loss="binary_crossentropy")
limpiador.fit(X_tr_ruido, X_tr, epochs=10, batch_size=256, verbose=0)     # entrada con ruido, respuesta limpia

mostrar([X_te_ruido[:8], limpiador.predict(X_te_ruido[:8], verbose=0), X_te[:8]],
        ["Con ruido (entrada)", "Limpiadas por la red", "Originales"])

**Qué deberías ver:** imágenes muy ruidosas que la red devuelve limpias y reconocibles. Se usa para restaurar fotos antiguas, escaneos o imágenes médicas.

### Ejercicio 3
Calcula el error cuadrático medio (MSE) entre las imágenes **con ruido** y las originales, y entre las **limpiadas** y las originales, sobre las 1.000 primeras de prueba. Guárdalos en `mse_ruido` y `mse_limpia`. ¿Cuánto bajó el error?

In [ ]:
mse_ruido = None   # TU CÓDIGO AQUÍ
mse_limpia = None  # TU CÓDIGO AQUÍ
print(mse_ruido, mse_limpia)

In [ ]:
revisar(lambda: np.isclose(mse_ruido, np.mean((X_te_ruido[:1000] - X_te[:1000]) ** 2), rtol=1e-4) and mse_limpia < mse_ruido)

## Paso 2.2. Detectar anomalías

Este es uno de los usos más aplicados de los autoencoders, y sirve para proyectos reales: **fraudes, fallas de máquinas, defectos de fabricación**.

**La idea:**

1. Entrenar el autoencoder **solo con casos normales**.
2. Aprende a reconstruir bien **lo normal**.
3. Cuando llega algo distinto, lo reconstruye **mal**: su **error de reconstrucción** es alto.
4. Ese error es una **señal de anomalía**.

**Nuestro caso:** una fábrica de calzado deportivo. Lo "normal" son las **zapatillas**. Queremos detectar cualquier otra cosa que aparezca en la línea de producción.

In [ ]:
NORMAL = 7                                           # Zapatilla
X_normal = X_tr[y_tr == NORMAL]
print("Zapatillas para entrenar:", len(X_normal))

keras.utils.set_random_seed(42)
detector = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(64, activation="relu"), layers.Dense(16, activation="relu"),
    layers.Dense(64, activation="relu"), layers.Dense(784, activation="sigmoid"),
])
detector.compile(optimizer="adam", loss="mse")
detector.fit(X_normal, X_normal, epochs=20, batch_size=128, verbose=0)

def error_reconstruccion(X):
    return np.mean((X - detector.predict(X, verbose=0)) ** 2, axis=1)     # un error por imagen

errores = error_reconstruccion(X_te)
es_anomalia = (y_te != NORMAL).astype(int)            # 1 si NO es zapatilla

plt.figure(figsize=(8, 4))
plt.hist(errores[es_anomalia == 0], bins=60, alpha=0.6, label="Zapatillas (normales)", density=True)
plt.hist(errores[es_anomalia == 1], bins=60, alpha=0.6, label="Otras prendas (anomalías)", density=True)
plt.xlabel("Error de reconstrucción"); plt.legend(); plt.title("Las anomalías se reconstruyen peor"); plt.show()

**Qué deberías ver:** las zapatillas con errores bajos, concentradas a la izquierda, y las otras prendas con errores más altos, a la derecha.

**¿Dónde poner el umbral?** Una regla común: el percentil 95 de los errores de los casos **normales**. Así, solo un 5% de las zapatillas buenas dispararía una falsa alarma.

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score

errores_normales_tr = error_reconstruccion(X_normal)
umbral = np.percentile(errores_normales_tr, 95)
alarma = (errores > umbral).astype(int)

print(f"Umbral (percentil 95 de las zapatillas de entrenamiento): {umbral:.4f}")
print(f"ROC AUC del error como detector: {roc_auc_score(es_anomalia, errores):.3f}")
print(f"Recall (anomalías detectadas): {recall_score(es_anomalia, alarma):.3f}")
print(f"Falsas alarmas en zapatillas: {alarma[es_anomalia == 0].mean():.3f}")

peores = np.argsort(errores)[::-1][:8]
mostrar([X_te[peores], detector.predict(X_te[peores], verbose=0)], ["Mayor error: lo más 'raro'", "Cómo las reconstruye"])
print("Qué son:", [PRENDAS[y_te[i]] for i in peores])

**Qué deberías ver:** un ROC AUC alto (sobre 0,9), la mayoría de las anomalías detectadas y cerca de un 5% de falsas alarmas, como fijamos con el percentil 95. En las imágenes con mayor error, el detector **intenta dibujar una zapatilla** sobre algo que no lo es: por eso el error es tan alto.

**Lo valioso:** el detector **nunca vio una anomalía** al entrenar. En fraude o fallas eso es clave, porque los casos anormales son pocos y cambian todo el tiempo.

### Ejercicio 4
¿Qué prenda es la **más difícil** de distinguir de una zapatilla? Calcula el **error de reconstrucción promedio** de cada prenda en `X_te` y guarda en `prenda_mas_parecida` el nombre de la prenda (distinta de Zapatilla) con **menor** error promedio.

In [ ]:
prenda_mas_parecida = None  # TU CÓDIGO AQUÍ
print(prenda_mas_parecida)

In [ ]:
_p = pd.Series(errores).groupby(y_te).mean().drop(NORMAL)
revisar(lambda: prenda_mas_parecida == PRENDAS[_p.idxmin()])

**Qué deberías ver:** casi siempre **Sandalia** o **Botín**: también son calzado. Tiene sentido.

### Ejercicio 5
Si la fábrica prefiere **no dejar pasar casi ninguna anomalía**, aunque haya más falsas alarmas, ¿el umbral debe subir o bajar? Prueba con el percentil **80** de las zapatillas de entrenamiento y guarda el nuevo recall en `recall_p80`.

In [ ]:
recall_p80 = None  # TU CÓDIGO AQUÍ
print(recall_p80)

In [ ]:
revisar(lambda: np.isclose(recall_p80, recall_score(es_anomalia, (errores > np.percentile(errores_normales_tr, 80)).astype(int))))

---
# Parte 3. GAN: la idea (láminas 91 a 104)

El autoencoder **reconstruye** imágenes que ya existen. Una **GAN** (red generativa adversaria) **crea imágenes nuevas** que nunca existieron.

## Paso 3.1. Falsificador contra detective

Una GAN tiene **dos redes que compiten**:

| Red | Rol | Recibe | Entrega | Quiere |
|---|---|---|---|---|
| **Generador** | El falsificador | Ruido al azar (un vector de 100 números) | Una imagen falsa | Engañar al discriminador |
| **Discriminador** | El detective | Una imagen, real o falsa | La probabilidad de que sea real | No dejarse engañar |

Entrenan **por turnos**, y cada uno mejora porque el otro mejora:

1. Al principio el generador dibuja manchas, y el discriminador aprende fácil a reconocerlas.
2. El generador ajusta sus pesos para que sus manchas **parezcan más reales** al discriminador.
3. El discriminador tiene que afinar su ojo para seguir distinguiendo.
4. Así, ronda a ronda, las imágenes falsas se vuelven cada vez más realistas.

**El ruido de entrada** (lámina 97) es la "semilla" de cada imagen: ruidos distintos producen prendas distintas.

## Paso 3.2. Las pérdidas (láminas 99 a 102)

Las dos redes usan entropía cruzada binaria (Clase 15), con objetivos opuestos:

**Discriminador:** quiere decir 1 a las reales y 0 a las falsas.

$$L_D = \text{BCE}(1, D(\text{real})) + \text{BCE}(0, D(G(z)))$$

**Generador:** quiere que el discriminador diga **1** a sus imágenes falsas.

$$L_G = \text{BCE}(1, D(G(z)))$$

donde $z$ es el ruido, $G(z)$ la imagen falsa y $D(\cdot)$ la probabilidad de "real" que entrega el discriminador.

El generador **nunca ve imágenes reales**: aprende solo a través de lo que el discriminador opina de sus falsificaciones.

### Ejercicio 6
Para fijar la idea de la pérdida, calcula a mano con `np.log` la entropía cruzada binaria $-[y \log(p) + (1-y)\log(1-p)]$ en dos casos:

- El discriminador recibe una imagen **falsa** ($y = 0$) y dice $p = 0{,}9$ (cree que es real). Guarda la pérdida en `bce_engañado`.
- Recibe una falsa y dice $p = 0{,}1$ (la descubre). Guarda la pérdida en `bce_acierta`.

In [ ]:
bce_engañado = None  # TU CÓDIGO AQUÍ
bce_acierta = None   # TU CÓDIGO AQUÍ
print(bce_engañado, bce_acierta)

In [ ]:
revisar(lambda: np.isclose(bce_engañado, -np.log(0.1)) and np.isclose(bce_acierta, -np.log(0.9)))

**Qué deberías ver:** 2,30 cuando se deja engañar y 0,11 cuando acierta. La pérdida castiga mucho los errores con confianza.

---
# Parte 4. GAN: verla funcionar (láminas 105 a 132)

Seguimos la estructura de las láminas, con Keras: definir el generador y el discriminador, las pérdidas, el paso de entrenamiento con `GradientTape` (Clase 14) y el ciclo. **En esta parte no hay ejercicios:** ejecuten las celdas en orden y observen cómo aparecen las prendas.

## Paso 4.1. Los datos

Para la GAN, las imágenes van de **−1 a 1** en vez de 0 a 1: así calzan con la salida `tanh` del generador.

In [ ]:
RUIDO = 100
LOTE = 128
X_gan = (X_tr.reshape(-1, 28, 28, 1) * 2.0 - 1.0).astype("float32")     # de [0, 1] a [-1, 1]
datos_gan = tf.data.Dataset.from_tensor_slices(X_gan).shuffle(60_000, seed=0).batch(LOTE, drop_remainder=True)
print("Lotes por época:", len(list(datos_gan)))

## Paso 4.2. El generador (lámina 111)

De un vector de ruido a una imagen de 28 × 28. Usamos capas **`Conv2DTranspose`**, que hacen lo contrario de una convolución con paso 2: **duplican** el tamaño de la imagen (7 → 14 → 28).

In [ ]:
keras.utils.set_random_seed(42)
generador = keras.Sequential([
    layers.Input(shape=(RUIDO,)),
    layers.Dense(7 * 7 * 64), layers.BatchNormalization(), layers.LeakyReLU(0.2),
    layers.Reshape((7, 7, 64)),                                                   # de vector a "imagen" pequeña
    layers.Conv2DTranspose(64, (4, 4), strides=2, padding="same"), layers.BatchNormalization(), layers.LeakyReLU(0.2),  # 14 x 14
    layers.Conv2DTranspose(1, (4, 4), strides=2, padding="same", activation="tanh"),                                   # 28 x 28
], name="generador")
generador.summary()

prueba = generador(tf.random.normal((8, RUIDO)), training=False)
mostrar([(prueba.numpy() + 1) / 2], ["El generador sin entrenar: puro ruido"])

**Qué deberías ver:** imágenes de 28 × 28 que son solo ruido gris. Todavía no aprendió nada.

## Paso 4.3. El discriminador (lámina 111)

Una CNN de clasificación binaria, como las de la Clase 15: recibe una imagen y entrega un número. Sin sigmoide al final: entrega un **logit**, y la pérdida aplica la sigmoide por dentro (`from_logits=True`), que es más estable numéricamente.

In [ ]:
discriminador = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(64, (4, 4), strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),    # 14 x 14
    layers.Conv2D(128, (4, 4), strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),   # 7 x 7
    layers.Flatten(),
    layers.Dense(1),                                                                                      # logit: real o falsa
], name="discriminador")
discriminador.summary()

## Paso 4.4. Pérdidas y optimizadores (lámina 124)

Las fórmulas del Paso 3.2, en código. Cada red tiene **su propio optimizador**, porque se entrenan por separado.

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)

def perdida_discriminador(logits_reales, logits_falsos):
    return bce(tf.ones_like(logits_reales), logits_reales) + bce(tf.zeros_like(logits_falsos), logits_falsos)

def perdida_generador(logits_falsos):
    return bce(tf.ones_like(logits_falsos), logits_falsos)      # quiere que digan "real" (1)

opt_g = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
opt_d = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

`tf.ones_like(x)` crea un tensor de unos con la misma forma que `x`: las etiquetas "real". `tf.zeros_like`, las etiquetas "falsa". Los valores `learning_rate=2e-4` y `beta_1=0.5` son los recomendados para GAN: las dos redes deben aprender a un ritmo parecido para que ninguna aplaste a la otra.

## Paso 4.5. Un paso de entrenamiento (láminas 125 y 126)

Es el ciclo de la Clase 14 con `GradientTape`, pero con **dos cintas**: una para cada red.

In [ ]:
@tf.function                                                  # compila la función para que sea mucho más rápida
def paso(imagenes_reales):
    ruido = tf.random.normal((LOTE, RUIDO))
    with tf.GradientTape() as cinta_g, tf.GradientTape() as cinta_d:
        falsas = generador(ruido, training=True)                         # 1. el generador falsifica
        logits_reales = discriminador(imagenes_reales, training=True)    # 2. el discriminador opina de las reales
        logits_falsos = discriminador(falsas, training=True)             #    y de las falsas
        loss_g = perdida_generador(logits_falsos)                        # 3. las dos pérdidas
        loss_d = perdida_discriminador(logits_reales, logits_falsos)
    grad_g = cinta_g.gradient(loss_g, generador.trainable_variables)     # 4. gradientes de cada red
    grad_d = cinta_d.gradient(loss_d, discriminador.trainable_variables)
    opt_g.apply_gradients(zip(grad_g, generador.trainable_variables))    # 5. cada red actualiza solo sus pesos
    opt_d.apply_gradients(zip(grad_d, discriminador.trainable_variables))
    return loss_g, loss_d

**Lo importante:** el gradiente del generador **atraviesa el discriminador** (porque su pérdida depende de lo que el discriminador opina), pero solo se actualizan los pesos del generador. Así el falsificador aprende qué cambiar para engañar mejor.

## Paso 4.6. Entrenar y mirar cómo mejora (lámina 127)

Usamos siempre **el mismo ruido** para mostrar el progreso: así vemos cómo evolucionan las mismas 8 "semillas". Con GPU, cada época tarda unos 10 a 20 segundos: unos 3 minutos en total. Aprovechen ese tiempo para leer la Parte 5.

In [ ]:
EPOCAS_GAN = 10
ruido_fijo = tf.random.normal((8, RUIDO), seed=1)
historial = {"generador": [], "discriminador": []}
muestras = {}

inicio = time.time()
for epoca in range(1, EPOCAS_GAN + 1):
    for lote in datos_gan:
        lg, ld = paso(lote)
    historial["generador"].append(float(lg)); historial["discriminador"].append(float(ld))
    if epoca in (1, 3, 8, EPOCAS_GAN):
        muestras[epoca] = (generador(ruido_fijo, training=False).numpy() + 1) / 2
    print(f"Época {epoca:2d} | pérdida G {float(lg):.3f} | pérdida D {float(ld):.3f} | {time.time() - inicio:.0f} s")

mostrar(list(muestras.values()), [f"Época {e}" for e in muestras])

**Qué deberías ver:** en la época 1, manchas; en la 3, siluetas; hacia la 10, prendas reconocibles: camisetas, pantalones, zapatos y bolsos **que no existen** en el dataset.

## Paso 4.7. Las pérdidas (lámina 128)

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(historial["generador"], label="Generador"); plt.plot(historial["discriminador"], label="Discriminador")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.legend(); plt.grid(alpha=0.3); plt.title("Pérdidas de la GAN"); plt.show()

**Cómo leerlas:** en una GAN, **las pérdidas no bajan hasta cero** como en un clasificador, porque las dos redes compiten. Lo sano es que **se estabilicen** en valores parecidos. Si la del discriminador llega a casi 0, ganó la competencia y el generador deja de aprender. Por eso, en GAN, la evaluación principal es **mirar las imágenes**.

**Prendas nuevas:** con ruido al azar, el generador crea prendas que no existen en el dataset.

In [ ]:
nuevas = (generador(tf.random.normal((16, RUIDO)), training=False).numpy() + 1) / 2
mostrar([nuevas[:8], nuevas[8:]])

## Paso 4.8. Moverse por el espacio de ruido

Como cada imagen sale de un vector de ruido, podemos ir **de un vector a otro de a poco** y ver cómo una prenda se transforma en otra. Es la prueba de que el generador aprendió algo continuo, no solo copió imágenes.

In [ ]:
z1, z2 = tf.random.normal((1, RUIDO), seed=3), tf.random.normal((1, RUIDO), seed=7)
pasos = tf.concat([(1 - a) * z1 + a * z2 for a in np.linspace(0, 1, 8)], axis=0)
mostrar([(generador(pasos, training=False).numpy() + 1) / 2], ["De un ruido a otro, en 8 pasos"])

---
# Parte 5. Hugging Face (Capítulo 10, láminas 12 a 20)

## Paso 5.1. ¿Qué es Hugging Face?

> **Hugging Face:** una plataforma para compartir modelos de IA ya entrenados, datasets y aplicaciones. Es "el GitHub de la IA".

Tiene tres secciones principales:

| Sección | Qué hay | Lo usamos en |
|---|---|---|
| **Models** | Más de un millón de modelos entrenados: texto, imágenes, audio | Hoy |
| **Datasets** | Datos listos para usar con una línea de código | Hoy (IMDb) |
| **Spaces** | Aplicaciones de demostración, muchas hechas con Gradio | Clase 6, donde publicaron su app |

## Paso 5.2. ¿Por qué preentrenados?

En la Clase 13, una LSTM entrenada con 60 reseñas perdió contra una regresión logística. En la Clase 16, con 25.000 reseñas, la alcanzó pero no la superó. Un modelo preentrenado **ya leyó miles de millones de palabras** antes de llegar a nosotros: ya sabe cómo funciona el idioma. Solo hay que usarlo, o ajustarlo un poco a nuestra tarea (eso es el *fine-tuning*, de la próxima clase).

| | Entrenar desde cero (Clases 13 a 16) | Usar un preentrenado (hoy) |
|---|---|---|
| Datos que necesita | Miles o millones de ejemplos | Ninguno para empezar |
| Tiempo | Minutos u horas de entrenamiento | Segundos para descargar |
| Conocimiento del idioma | Solo el de nuestros datos | El de enormes corpus de texto |

## Paso 5.3. Cómo leer la ficha de un modelo

Abran [huggingface.co/models](https://huggingface.co/models) y busquen `nlptown/bert-base-multilingual-uncased-sentiment`. Cada modelo tiene una **ficha** (*model card*). Lo que hay que mirar antes de usarlo:

| Qué mirar | Dónde | Por qué importa |
|---|---|---|
| **Tarea** | Etiqueta arriba (por ejemplo, *Text Classification*) | Dice qué hace el modelo |
| **Idiomas** | Etiquetas y descripción | Un modelo entrenado solo en inglés funciona mal en español |
| **Descargas y "me gusta"** | Arriba a la derecha | Señal de que otros lo usan y funciona |
| **Datos de entrenamiento** | Descripción | Con qué textos aprendió: define en qué será bueno |
| **Licencia** | Barra lateral | Si se puede usar en un proyecto, también comercial |
| **Tamaño** | Pestaña *Files* | Un modelo de 5 GB puede no caber en Colab gratuito |
| **Widget de prueba** | A la derecha | Para probarlo en el navegador antes de descargarlo |

Este modelo, por ejemplo, entrega de **1 a 5 estrellas** y fue entrenado con reseñas de productos en seis idiomas, entre ellos el español. Justo lo que necesitamos para las reseñas de Samsung.

## Paso 5.4. Instalar y preparar

In [ ]:
!pip install -q transformers datasets sentencepiece

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

DISPOSITIVO = 0 if torch.cuda.is_available() else -1      # 0 = primera GPU; -1 = CPU
print("Usando:", "GPU" if DISPOSITIVO == 0 else "CPU")

---
# Parte 6. Pipeline API (láminas 21 a 39)

> **`pipeline`:** la forma más simple de usar un modelo de Hugging Face. En una línea descarga el modelo y su tokenizador, y en otra lo aplica a un texto.

```python
clasificador = pipeline("tarea", model="usuario/modelo")
clasificador("un texto")
```

## Paso 6.1. Qué pasa por dentro (láminas 24 y 25)

Un pipeline hace tres pasos, los mismos que hicimos a mano en las Clases 12 y 13:

| Paso | Qué hace | Lo vimos en |
|---|---|---|
| 1. **Tokenizar** | Divide el texto en tokens y los convierte en números | Codificación entera, Clase 12 |
| 2. **Modelo** | La red procesa los números y entrega puntajes (*logits*) | Clases 14 a 16 |
| 3. **Posprocesar** | Convierte los puntajes en probabilidades (softmax) y en etiquetas | Softmax, Clase 15 |

**Qué vamos a hacer:** recorrer los tres pasos a mano con el modelo de estrellas.

In [ ]:
MODELO_ESTRELLAS = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizador = AutoTokenizer.from_pretrained(MODELO_ESTRELLAS)
modelo_estrellas = AutoModelForSequenceClassification.from_pretrained(MODELO_ESTRELLAS)

texto = "La batería dura muy poco y la pantalla se raya fácil."

# 1. Tokenizar
entrada = tokenizador(texto, return_tensors="pt")
print("Tokens:", tokenizador.convert_ids_to_tokens(entrada["input_ids"][0]))
print("Números:", entrada["input_ids"][0].tolist())

**Qué deberías ver:** el texto dividido en **tokens**, algunos con `##` (por ejemplo `bat`, `##eria`). Los modelos modernos no usan palabras completas sino **subpalabras**: así manejan palabras que nunca vieron, armándolas por partes. Al inicio y al final aparecen `[CLS]` y `[SEP]`, marcas especiales del modelo (como el `<s>` de los n-gramas de la Clase 12).

In [ ]:
# 2. Modelo: entrega un logit por clase
with torch.no_grad():                                # no calcular gradientes: solo predecimos
    logits = modelo_estrellas(**entrada).logits
print("Logits:", logits[0].numpy().round(2))

# 3. Posprocesar: softmax y la clase más probable
probabilidades = torch.softmax(logits, dim=-1)[0].numpy()
etiquetas = modelo_estrellas.config.id2label
for i, p in enumerate(probabilidades):
    print(f"{etiquetas[i]}: {p:.3f}")

# Lo mismo, en una línea con pipeline
estrellas = pipeline("text-classification", model=MODELO_ESTRELLAS, device=DISPOSITIVO)
print("\nCon pipeline:", estrellas(texto))

**Qué deberías ver:** 5 logits, 5 probabilidades (de "1 star" a "5 stars") con la más alta en 1 o 2 estrellas, y el pipeline entregando la misma respuesta en una línea.

### Ejercicio 7
¿En cuántos tokens divide el tokenizador la frase de `frase_tokens`? Guárdalo en `n_tokens`, **contando** `[CLS]` y `[SEP]`.

In [ ]:
frase_tokens = "El reloj inteligente monitorea el sueño perfectamente."
n_tokens = None  # TU CÓDIGO AQUÍ
print(n_tokens)

In [ ]:
revisar(lambda: n_tokens == len(tokenizador(frase_tokens)["input_ids"]))

### Ejercicio 8
Calcula **a mano**, con NumPy, el softmax de los `logits` del Paso 6.1 y guarda la probabilidad de la clase **más probable** en `prob_max`. Debe coincidir con el `score` del pipeline.

Recuerda: softmax(z)ₖ = e^(zₖ) / Σ e^(zⱼ).

In [ ]:
z = logits[0].numpy()
prob_max = None  # TU CÓDIGO AQUÍ
print(prob_max)

In [ ]:
revisar(lambda: np.isclose(prob_max, estrellas(texto)[0]["score"], atol=1e-3))

## Paso 6.2. Un recorrido por tareas (láminas 26 a 39)

Con la misma línea de código, cambiando la tarea y el modelo, se resuelven problemas muy distintos. Elegimos modelos que funcionan en **español**. La primera vez, cada uno se descarga (unos segundos).

### Sentimiento en tres clases

In [ ]:
sentimiento = pipeline("text-classification", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student", device=DISPOSITIVO)
frases = ["Me encanta este reloj, lo uso todos los días.",
          "No funciona, lo devolví al segundo día.",
          "Llegó el martes en una caja azul."]
for f, r in zip(frases, sentimiento(frases)):
    print(f"{r['label']:8s} {r['score']:.2f} | {f}")

### Clasificación sin entrenamiento (*zero-shot*)

**Lo más útil para proyectos:** clasificar en **categorías que elegimos nosotros**, sin ningún dato de entrenamiento. El modelo evalúa qué tan bien encaja el texto con cada etiqueta.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=DISPOSITIVO)

reclamo = "Pagué hace tres semanas y el pedido todavía no llega a mi casa."
categorias = ["despacho", "calidad del producto", "atención al cliente", "precio"]
r = zero_shot(reclamo, candidate_labels=categorias, hypothesis_template="Este mensaje trata sobre {}.")
for etiqueta, puntaje in zip(r["labels"], r["scores"]):
    print(f"{etiqueta:22s} {puntaje:.3f}")

**Qué deberías ver:** "despacho" con el puntaje más alto. Nadie entrenó al modelo con esas categorías: las entiende porque entiende el idioma.

### Ejercicio 9
Clasifica `mi_mensaje` con `zero_shot` en las categorías `["felicitación", "reclamo", "pregunta"]` y guarda la etiqueta ganadora en `etiqueta_ganadora`.

In [ ]:
mi_mensaje = "¿Hasta qué hora atienden el sábado en la sucursal de Ñuñoa?"
etiqueta_ganadora = None  # TU CÓDIGO AQUÍ
print(etiqueta_ganadora)

In [ ]:
revisar(lambda: etiqueta_ganadora == zero_shot(mi_mensaje, candidate_labels=["felicitación", "reclamo", "pregunta"],
                                                 hypothesis_template="Este mensaje es una {}.")["labels"][0]
        or etiqueta_ganadora == zero_shot(mi_mensaje, candidate_labels=["felicitación", "reclamo", "pregunta"])["labels"][0])

### Reconocimiento de entidades (NER)

Encuentra **personas, lugares y organizaciones** en un texto. Útil para procesar noticias, contratos o reclamos.

In [ ]:
ner = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", aggregation_strategy="simple", device=DISPOSITIVO)
noticia = "Samsung y la ONG Innovacien lanzaron en Santiago un curso de inteligencia artificial dirigido por David Leal."
for e in ner(noticia):
    print(f"{e['entity_group']:5s} {e['score']:.2f} | {e['word']}")

**Qué deberías ver:** organizaciones (ORG) como Samsung e Innovacien, un lugar (LOC) como Santiago y una persona (PER).

### Traducción

In [ ]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en", device=DISPOSITIVO)
print(traductor("La batería dura casi dos días y la pantalla se ve perfecta al sol.")[0]["translation_text"])

### Completar el texto (*fill-mask*): BETO, un modelo chileno

BETO es un modelo de lenguaje en español entrenado por investigadores de la **Universidad de Chile**. Su tarea original es adivinar una palabra oculta, marcada con `[MASK]`: es el modelo de lenguaje de la Clase 12, pero mirando las palabras de **ambos lados**.

In [ ]:
completar = pipeline("fill-mask", model="dccuchile/bert-base-spanish-wwm-cased", device=DISPOSITIVO)
for r in completar("La capital de Chile es [MASK]."):
    print(f"{r['token_str']:12s} {r['score']:.3f}")

### Ejercicio 10
Usa `completar` con la frase `"El reloj tiene una [MASK] que dura dos días."` y guarda en `palabra_top` la palabra más probable (`token_str` del primer resultado).

In [ ]:
palabra_top = None  # TU CÓDIGO AQUÍ
print(palabra_top)

In [ ]:
revisar(lambda: palabra_top == completar("El reloj tiene una [MASK] que dura dos días.")[0]["token_str"])

### Generar texto con un modelo de lenguaje pequeño

Los modelos tipo ChatGPT **generan** texto: predicen la siguiente palabra (en realidad, el siguiente token) una y otra vez, como el modelo de n-gramas de la Clase 12 pero con una red enorme. Probamos uno pequeño (500 millones de parámetros) que cabe en Colab.

In [ ]:
generador_texto = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=DISPOSITIVO)
mensajes = [{"role": "user", "content": "Explica en dos oraciones qué es un autoencoder, para un estudiante."}]
respuesta = generador_texto(mensajes, max_new_tokens=100, do_sample=False)
print(respuesta[0]["generated_text"][-1]["content"])

**Qué deberías ver:** una explicación razonable, quizás con algún error: es un modelo pequeño. Los grandes (ChatGPT, Claude, Gemini) tienen cientos de veces más parámetros. La próxima clase veremos cómo **ajustar** (*fine-tuning*) un modelo así a una tarea propia.

---
# Parte 7. Preentrenado contra lo nuestro

## Paso 7.1. IMDb: el modelo preentrenado contra la LSTM de la Clase 16

En la Clase 16, con 25.000 reseñas de entrenamiento, la mejor red llegó a cerca de **0,88** de exactitud, igual que la regresión logística. Probemos un modelo de Hugging Face **ya ajustado** para reseñas de películas, sin entrenar nada.

Para que sea rápido usamos 500 reseñas de prueba al azar, con texto original (no codificado como números). `truncation=True` corta las reseñas más largas que el máximo del modelo (512 tokens).

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=42).select(range(500))
textos_imdb, reales_imdb = imdb["text"], np.array(imdb["label"])          # 1 = positiva, 0 = negativa
print("Reseñas:", len(textos_imdb), "| Positivas:", reales_imdb.mean())

clasif_imdb = pipeline("text-classification", model="lvwerra/distilbert-imdb", device=DISPOSITIVO, truncation=True)
inicio = time.time()
salidas_imdb = clasif_imdb(textos_imdb, batch_size=32)
print(f"Tiempo: {time.time() - inicio:.0f} s | Ejemplo de salida: {salidas_imdb[0]}")

### Ejercicio 11
Convierte las salidas en predicciones 0 o 1 (la etiqueta positiva es `"POSITIVE"`; si en tu versión aparece `"LABEL_1"`, esa es la positiva), calcula la exactitud contra `reales_imdb` y guárdala en `exac_preentrenado`. ¿Cuánto más que el 0,88 de la Clase 16?

In [ ]:
from sklearn.metrics import accuracy_score

exac_preentrenado = None  # TU CÓDIGO AQUÍ
print(exac_preentrenado)

In [ ]:
revisar(lambda: np.isclose(exac_preentrenado, accuracy_score(reales_imdb, [1 if s["label"] in ("POSITIVE", "LABEL_1") else 0 for s in salidas_imdb])))

**Qué deberías ver:** cerca de **0,93**: unos 5 puntos más que nuestra mejor red, **sin entrenar nada** y en segundos. Este modelo partió de DistilBERT, que ya conocía el inglés, y lo ajustaron con las reseñas de IMDb. Esa combinación (preentrenar con mucho texto general y ajustar con pocos datos de la tarea) es la que domina hoy.

## Paso 7.2. Las reseñas de Samsung: estrellas y temas

Volvemos a las reseñas de productos Samsung de las Clases 11 a 13 (desde la API, o el respaldo ficticio si no responde).

In [ ]:
import requests

URL_RESENAS = "https://apps.bazaarvoice.com/bfd/v1/clients/samsung-es_cl/api-products/cv2/resources/data/reviews.json"
HEADERS_API = {"Accept": "application/json", "X-Bazaarvoice-Api-Version": "5.5", "X-Bazaarvoice-Platform-Version": "2",
               "Bv-Bfd-Token": "17499,main_site,es_CL", "Origin": "https://www.samsung.com"}

def descargar_resenas(skus=("SM-L705FZB1CHO", "SM-L320NZSALTA", "SM-R390NZAALTA")):
    todas = []
    for sku in skus:
        parametros = {"resource": "reviews", "action": "REVIEWS_N_STATS",
                      "filter": [f"productid:eq:{sku}", "contentlocale:eq:es_CL,es*"],
                      "apiversion": "5.5", "displaycode": "17499-es_cl", "limit": 100, "offset": 0}
        res = requests.get(URL_RESENAS, params=parametros, headers=HEADERS_API, timeout=15)
        res.raise_for_status()
        todas.extend(res.json().get("response", {}).get("Results", [])); time.sleep(1)
    return pd.DataFrame(todas)

RESPALDO = [(5, 'Excelente compañero para correr', 'Lo uso todos los días para correr y el GPS es muy preciso. La batería dura casi dos días con el monitoreo activado.'), (5, 'Me encanta el seguimiento del sueño', 'El análisis del sueño es muy completo y me ayudó a ordenar mis horarios. La pantalla se ve perfecta incluso al sol.'), (4, 'Muy buen reloj, batería justa', 'Cumple con todo lo que promete, las notificaciones llegan rápido. La batería dura un día y medio, esperaba un poco más.'), (5, 'Pantalla espectacular', 'La pantalla es brillante y nítida, se lee perfecto bajo el sol. Muy cómodo para usar todo el día.'), (1, 'La batería no dura nada', 'Con el GPS activado la batería no alcanza a durar un día completo. Muy decepcionado para el precio que tiene.'), (2, 'Problemas de conexión con el celular', 'Se desconecta del celular varias veces al día y pierdo las notificaciones. Tuve que reinstalar la aplicación dos veces.'), (5, 'Ideal para el gimnasio', 'Registra bien los entrenamientos y el ritmo cardíaco. La aplicación muestra todo muy claro.'), (4, 'Buen producto, despacho rápido', 'Llegó en dos días y bien embalado. El reloj funciona muy bien, aunque la correa es un poco dura al principio.'), (1, 'Dejó de cargar al mes', 'Al mes de uso dejó de cargar. El servicio técnico se demoró tres semanas en responder. No lo recomiendo.'), (5, 'El mejor reloj que he tenido', 'Diseño elegante, pantalla grande y la batería dura bastante. Las funciones de salud son muy útiles.'), (3, 'Cumple, pero es caro', 'Funciona bien, pero por el precio esperaba más autonomía de batería. La pantalla es muy buena.'), (2, 'La aplicación es lenta', 'La aplicación del celular tarda mucho en sincronizar los datos del sueño. A veces no sincroniza nada.'), (5, 'Perfecto para monitorear la salud', 'Mide el ritmo cardíaco, el oxígeno y el estrés. Me da tranquilidad poder revisar mi salud cada día.'), (4, 'Muy cómodo', 'Casi no se siente en la muñeca y se puede dormir con él. La batería podría durar más.'), (1, 'Pantalla rayada en una semana', 'La pantalla se rayó a la semana de uso normal. Esperaba un vidrio más resistente.'), (5, 'Excelente calidad precio', 'Lo compré en oferta y es una maravilla. Batería, pantalla y aplicación funcionan muy bien.'), (2, 'El GPS falla', 'El GPS pierde la señal en la ciudad y los recorridos quedan cortados. Para correr no me sirve.'), (5, 'Notificaciones perfectas', 'Recibo los mensajes y llamadas sin sacar el celular. La conexión con el teléfono es estable.'), (4, 'Buen seguimiento del sueño', 'El seguimiento del sueño es bastante preciso. Me gustaría que la aplicación diera más consejos.'), (1, 'No reconoce el celular', 'Después de la actualización ya no se conecta con mi celular. El soporte no me dio solución.'), (5, 'Muy buena batería', 'Con uso normal la batería me dura casi dos días completos. Carga rápido.'), (3, 'Correa incómoda', 'El reloj es bueno pero la correa que trae es incómoda y me irrita la piel. Tuve que comprar otra.'), (5, 'Lo recomiendo', 'Funciona perfecto, la pantalla es hermosa y el seguimiento del ejercicio es muy completo.'), (2, 'Mide mal el ritmo cardíaco', 'El ritmo cardíaco marca valores muy distintos a los de mi otro dispositivo. No confío en la medición.'), (4, 'Bonito y funcional', 'Se ve muy bien y cumple. La batería es lo único que podría mejorar.'), (5, 'Gran regalo', 'Se lo regalé a mi papá y lo usa todos los días para caminar. La pantalla es grande y fácil de leer.'), (1, 'Muy caro para lo que ofrece', 'Por ese precio esperaba una batería mucho mejor. Tengo que cargarlo todas las noches.'), (5, 'Excelente para nadar', 'Lo uso en la piscina sin problemas y registra bien las vueltas. Resistente al agua de verdad.'), (4, 'Buena aplicación', 'La aplicación es completa y fácil de usar. A veces la sincronización tarda un poco.'), (2, 'Se reinicia solo', 'Se reinicia solo varias veces a la semana. Lo envié al servicio técnico y sigue igual.'), (5, 'Pantalla siempre encendida', 'La pantalla siempre encendida es muy útil y la batería igual aguanta todo el día.'), (3, 'Bueno, sin más', 'Hace lo básico bien. Nada que me sorprenda, pero tampoco me ha fallado.'), (5, 'Mide muy bien el sueño', 'Me sorprendió lo bien que mide las fases del sueño. Ahora duermo mejor.'), (1, 'Despacho tardío', 'El despacho se demoró tres semanas y nadie respondía los correos. El producto está bien, la atención pésima.'), (4, 'Liviano y rápido', 'El reloj es liviano y la interfaz es muy rápida. La batería es justa si se usa el GPS.'), (5, 'Funciones de salud increíbles', 'El electrocardiograma y la presión son funciones muy útiles. Excelente para cuidar la salud.'), (2, 'Batería se agota rápido', 'Desde la última actualización la batería se agota en medio día. Antes duraba mucho más.'), (5, 'Muy satisfecha', 'Pantalla nítida, buena batería y se conecta sin problemas con mi celular.'), (4, 'Recomendable para correr', 'El GPS es preciso y el ritmo cardíaco es confiable. La correa podría ser más cómoda.'), (1, 'Llegó defectuoso', 'Llegó con la pantalla con un pixel muerto. El cambio tomó más de un mes.'), (5, 'Super cómodo para dormir', 'No molesta para dormir y el análisis del sueño es muy bueno. La batería aguanta bien.'), (3, 'Pantalla buena, batería regular', 'La pantalla es excelente pero la batería no me dura el día completo con todo activado.'), (5, 'Me ayudó a moverme más', 'Los recordatorios para moverme y el seguimiento de pasos me motivaron a hacer más ejercicio.'), (2, 'La conexión bluetooth falla', 'Se pierde la conexión bluetooth con el celular constantemente. Muy frustrante.'), (4, 'Muy buen diseño', 'El diseño es elegante y la pantalla se ve muy bien. La aplicación podría ser más intuitiva.'), (5, 'Excelente para entrenar', 'Registra todo tipo de ejercicio y el ritmo cardíaco es exacto. Muy contento con la compra.'), (1, 'Servicio técnico pésimo', 'El reloj falló y el servicio técnico no me dio solución en dos meses. No vuelvo a comprar.'), (5, 'Batería de dos días', 'La batería dura dos días completos con uso normal. Muy contento con la autonomía.'), (4, 'Sincroniza bien', 'Se sincroniza bien con la aplicación del celular y las notificaciones llegan al instante.'), (2, 'Correa se rompió', 'La correa se rompió al segundo mes. Para un producto de este precio es inaceptable.'), (5, 'Pantalla y salud excelentes', 'La pantalla es preciosa y las funciones de salud son muy completas. Lo uso para controlar mi presión.'), (3, 'Precio alto', 'Es un buen reloj, pero el precio es demasiado alto comparado con otros modelos.'), (5, 'Perfecto', 'Todo funciona perfecto: batería, pantalla, sueño y ejercicio. Muy recomendable.'), (1, 'No sincroniza el sueño', 'Los datos del sueño nunca aparecen en la aplicación. Probé de todo y no funciona.'), (4, 'Muy útil en el día a día', 'Uso las notificaciones y el seguimiento de pasos todo el día. La batería alcanza justo.'), (5, 'Calidad premium', 'Materiales de muy buena calidad y una pantalla excelente. Vale lo que cuesta.'), (2, 'Mediciones poco confiables', 'Los pasos y las calorías no coinciden con la realidad. No confío en los datos.'), (5, 'Buenísimo para caminar', 'Lo uso para caminar todos los días. El GPS y los pasos son muy precisos.'), (4, 'Cumple lo prometido', 'Funciona bien y la batería dura lo que dicen. La aplicación tiene muchas opciones.'), (1, 'Pantalla se apaga sola', 'La pantalla se apaga sola y hay que reiniciarlo. Muy mala experiencia.')]

try:
    resenas = (descargar_resenas().drop_duplicates(subset="Id")
               .rename(columns={"Rating": "nota", "Title": "titulo", "ReviewText": "texto"})[["nota", "titulo", "texto"]])
    if len(resenas) < 30:
        raise ValueError
    FUENTE = "API de Samsung"
except Exception:
    resenas = pd.DataFrame(RESPALDO, columns=["nota", "titulo", "texto"])
    FUENTE = "respaldo ficticio"
resenas = resenas.dropna(subset=["texto"]).reset_index(drop=True)
resenas["completo"] = resenas["titulo"].fillna("") + ". " + resenas["texto"]
print("Fuente:", FUENTE, "| Reseñas:", len(resenas))

**Primera pregunta: ¿el modelo adivina la nota?** El modelo de estrellas del Paso 6.1 nunca vio estas reseñas. Le pedimos que prediga de 1 a 5 estrellas y comparamos con la nota real que puso cada cliente.

In [ ]:
pred_estrellas = [int(r["label"][0]) for r in estrellas(resenas["completo"].tolist(), truncation=True, batch_size=32)]
resenas["estrellas_modelo"] = pred_estrellas

print(pd.crosstab(resenas["nota"], resenas["estrellas_modelo"], rownames=["Nota real"], colnames=["Predicha"]))
print(f"\nExactitud exacta: {(resenas['nota'] == resenas['estrellas_modelo']).mean():.2f}")

**Qué deberías ver:** una matriz con la mayoría cerca de la diagonal. Acertar la estrella exacta es difícil (entre un 4 y un 5 hay poca diferencia), así que la métrica más justa es acertar **con un margen de una estrella**.

### Ejercicio 12
Calcula la proporción de reseñas donde la predicción queda **a una estrella o menos** de la nota real, y guárdala en `acierto_mas_menos_1`. Pista: `np.abs(...) <= 1`.

In [ ]:
acierto_mas_menos_1 = None  # TU CÓDIGO AQUÍ
print(acierto_mas_menos_1)

In [ ]:
revisar(lambda: np.isclose(acierto_mas_menos_1, (np.abs(resenas["nota"] - resenas["estrellas_modelo"]) <= 1).mean()))

**Qué deberías ver:** cerca de 0,8 o 0,9. Sin entrenamiento, el modelo entiende bastante bien si un cliente quedó contento.

**Segunda pregunta: ¿de qué hablan?** En la Clase 11 encontramos temas con K-medias y tuvimos que **adivinar** qué era cada grupo mirando sus palabras clave. Con *zero-shot*, **elegimos los temas** de antemano y el modelo clasifica.

In [ ]:
TEMAS = ["batería", "pantalla", "conexión con el celular", "salud y ejercicio", "despacho y servicio técnico", "precio"]
resultados_temas = zero_shot(resenas["completo"].tolist(), candidate_labels=TEMAS,
                             hypothesis_template="Esta reseña habla de {}.", batch_size=16)
resenas["tema"] = [r["labels"][0] for r in resultados_temas]

resumen = resenas.groupby("tema").agg(reseñas=("nota", "size"), nota_promedio=("nota", "mean")).round(2)
display(resumen.sort_values("nota_promedio"))
for tema in resumen.sort_values("nota_promedio").index[:1]:
    print(f"\nEjemplos del tema peor evaluado ({tema}):")
    for t in resenas.loc[resenas["tema"] == tema, "completo"].head(3):
        print(" -", t[:120])

**Qué deberías ver:** cuántas reseñas habla de cada tema y su nota promedio, y ejemplos del tema peor evaluado. Comparen con la Clase 11: allá los temas los **descubrimos** (útil cuando no sabemos qué buscar); aquí los **definimos** (útil cuando sabemos qué preguntar). Las dos herramientas se complementan.

---
# Cierre

| Parte | Lo esencial |
|---|---|
| Autoencoder | Comprime y reconstruye. Sirve para quitar ruido y, sobre todo, **detectar anomalías** con el error de reconstrucción. |
| GAN | Generador contra discriminador. Muy influyente, pero hoy la generación de imágenes usa otros modelos (de difusión). |
| Hugging Face | El repositorio de modelos preentrenados. Revisar tarea, idioma, licencia y tamaño en la ficha. |
| Pipeline | Tokenizar, modelo y posprocesar, en una línea. |
| Preentrenado | Sin entrenar nada, superó a nuestras mejores redes en IMDb (cerca de 0,93 contra 0,88). |

**Para el capstone:**

- **Si su proyecto usa texto**, partan por un pipeline de Hugging Face antes de entrenar algo propio. Si clasifican en categorías propias, prueben *zero-shot*.
- **Si usan imágenes**, también hay pipelines (`"image-classification"`, `"object-detection"`) y transfer learning (Capítulo 12).
- **Revisen la licencia** del modelo que elijan y menciónenlo en el informe.

**Próxima clase:** fine-tuning (ajustar un modelo preentrenado a una tarea propia), LoRA y cuantización (Capítulo 10, Unidades 2 a 4).

Guarden el notebook en `ejercicios_estudiantes` como `clase17_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
784 pixeles en 32 números: 24,5 veces más pequeño.

In [ ]:
factor_compresion = 784 / CODIGO
print(factor_compresion)

### Ejercicio 2
El codificador es un modelo por sí solo: recibe imágenes de 784 pixeles y entrega códigos de 32 números.

In [ ]:
codigos_8 = codificador.predict(X_te[:8], verbose=0)
print(codigos_8.shape)
print(codigos_8[0].round(2))

### Ejercicio 3

In [ ]:
limpiadas = limpiador.predict(X_te_ruido[:1000], verbose=0)
mse_ruido = np.mean((X_te_ruido[:1000] - X_te[:1000]) ** 2)
mse_limpia = np.mean((limpiadas - X_te[:1000]) ** 2)
print(round(mse_ruido, 4), round(mse_limpia, 4), "| el error baja a", round(mse_limpia / mse_ruido, 2), "veces el original")

### Ejercicio 4
`groupby(y_te)` agrupa los errores por prenda; `drop(NORMAL)` quita las zapatillas; `idxmin` da el número de la prenda con menor error.

In [ ]:
por_prenda = pd.Series(errores).groupby(y_te).mean().drop(NORMAL)
prenda_mas_parecida = PRENDAS[por_prenda.idxmin()]
print(prenda_mas_parecida)
print(por_prenda.rename(index=dict(enumerate(PRENDAS))).sort_values().round(4))

### Ejercicio 5
Para detectar más anomalías, el umbral debe **bajar**: más imágenes superan el umbral y se marcan. El percentil 80 está más abajo que el 95, así que sube el recall y también las falsas alarmas (cerca de un 20% de las zapatillas).

In [ ]:
umbral_80 = np.percentile(errores_normales_tr, 80)
alarma_80 = (errores > umbral_80).astype(int)
recall_p80 = recall_score(es_anomalia, alarma_80)
print(round(recall_p80, 3), "| falsas alarmas:", round(alarma_80[es_anomalia == 0].mean(), 3))

### Ejercicio 6
Con y = 0, la fórmula queda −log(1 − p). Engañado: −log(0,1) = 2,30. Acierta: −log(0,9) = 0,11.

In [ ]:
bce_engañado = -np.log(1 - 0.9)
bce_acierta = -np.log(1 - 0.1)
print(round(bce_engañado, 3), round(bce_acierta, 3))

### Ejercicio 7
El tokenizador devuelve un diccionario; `input_ids` es la lista de números, uno por token, incluidos `[CLS]` y `[SEP]`.

In [ ]:
ids = tokenizador(frase_tokens)["input_ids"]
n_tokens = len(ids)
print(n_tokens, tokenizador.convert_ids_to_tokens(ids))

### Ejercicio 8
Se resta el máximo antes de la exponencial para evitar números enormes; el resultado es el mismo.

In [ ]:
e = np.exp(z - z.max())
probs = e / e.sum()
prob_max = float(probs.max())
print(round(prob_max, 3), "| pipeline:", round(estrellas(texto)[0]["score"], 3))

### Ejercicio 9
Lo más probable es "pregunta".

In [ ]:
r = zero_shot(mi_mensaje, candidate_labels=["felicitación", "reclamo", "pregunta"], hypothesis_template="Este mensaje es una {}.")
etiqueta_ganadora = r["labels"][0]
print(etiqueta_ganadora, [round(s, 3) for s in r["scores"]])

### Ejercicio 10
Lo más probable es "batería".

In [ ]:
opciones = completar("El reloj tiene una [MASK] que dura dos días.")
palabra_top = opciones[0]["token_str"]
print(palabra_top, [(o["token_str"], round(o["score"], 3)) for o in opciones])

### Ejercicio 11

In [ ]:
pred_imdb = [1 if s["label"] in ("POSITIVE", "LABEL_1") else 0 for s in salidas_imdb]
exac_preentrenado = accuracy_score(reales_imdb, pred_imdb)
print(round(exac_preentrenado, 3), "| mejora sobre 0,88:", round(exac_preentrenado - 0.88, 3))

### Ejercicio 12

In [ ]:
acierto_mas_menos_1 = (np.abs(resenas["nota"] - resenas["estrellas_modelo"]) <= 1).mean()
print(round(acierto_mas_menos_1, 3))